# Notebook 1 - Hiểu dữ liệu và chia tập train / val / test (Bước 1 - 3)

Notebook này làm 3 bước đầu của project dự đoán vỡ nợ trên dữ liệu Lending Club:

1. Tóm tắt yêu cầu đề bài và lập từ điển giải thích toàn bộ các feature
2. Nạp dữ liệu gốc và quan sát ban đầu
3. EDA phục vụ việc chia train / val / test theo thời gian (kiểu Arturo-GA, tỷ lệ 70 / 15 / 15)

**Quy tắc làm việc:** mỗi hàm nằm ở một ô code riêng, mỗi kết quả nằm ở một ô code riêng. Phần markdown chỉ giải thích *làm gì* và *vì sao*. Mọi con số và mọi nhận xét về dữ liệu đều do ô code ngay bên dưới tính và in ra, không gõ tay.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
SEED = 42
ROOT = Path("..")
RAW_FILE = ROOT / "data/raw/accepted_2007_to_2018Q4.csv.gz"
OUT_DIR = ROOT / "data/processed"
ART_DIR = ROOT / "artifacts"
FIG_DIR = ART_DIR / "figures"
for folder in [OUT_DIR, ART_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError("Chưa có file dữ liệu. Tải accepted_2007_to_2018Q4.csv.gz từ Kaggle "
                            "(wordsforthewise/lending-club) và đặt vào data/raw/")
print("Tìm thấy file:", RAW_FILE.name)

---
# Bước 1 - Tóm tắt yêu cầu đề bài

## 1.1 Đề bài nói gì về topic này?
Đề bài nằm trong file Excel của môn học. Ô code bên dưới đọc sheet danh sách topic và in ra dòng nói về Lending Club, để phần tóm tắt có chỗ dựa thay vì chép lại theo trí nhớ.

In [ ]:
brief_file = next(ROOT.glob("*.xlsx"))
topics = pd.read_excel(brief_file, sheet_name="Topics")
print("File đề bài:", brief_file.name)

In [ ]:
lending_row = topics[topics["Project names"].str.contains("Lending Club", na=False)]
lending_row[["No", "Project names", "Topic", "Link dataset"]]

## 1.2 Bài toán của project

| Mục | Nội dung |
|---|---|
| Loại bài toán | Phân loại nhị phân (credit scoring): dự đoán một khoản vay có vỡ nợ hay không |
| Dữ liệu | Các khoản vay đã được Lending Club chấp nhận, giai đoạn 2007 - 2018 |
| Đầu vào | Thông tin biết được **tại thời điểm xét duyệt**: số tiền vay, kỳ hạn, lãi suất, hạng tín dụng, thu nhập, việc làm, thông tin hồ sơ tín dụng... |
| Đầu ra | Xác suất khoản vay bị vỡ nợ (từ 0 đến 1) |
| Nhãn | `1` nếu `loan_status` là `Charged Off`, `0` nếu là `Fully Paid` |
| Cách chia dữ liệu | Theo thời gian, 70% khoản vay cũ nhất để train, 15% tiếp theo để validation, 15% mới nhất để test |

Việc chọn đúng hai trạng thái làm nhãn và các nhóm trạng thái bị loại sẽ được **kiểm chứng bằng code ở Bước 2**, không chấp nhận chỉ vì đề bài hay một project khác làm vậy.

## 1.3 Từ điển dữ liệu cho toàn bộ feature
File gốc có rất nhiều cột nên cần một bảng giải thích trước khi làm gì khác. Bảng gồm: tên cột, nhóm chủ đề, ý nghĩa, và một **giả thuyết** là cột đó có biết được lúc nộp đơn hay không.

Hai lưu ý quan trọng:
- Phần *ý nghĩa* được viết từ tên cột và hiểu biết chung về cho vay, vì bộ dữ liệu Kaggle không kèm file mô tả cột. Dữ liệu không tự chứng minh được định nghĩa, nhưng ở Bước 2 mỗi dòng của từ điển sẽ được đối chiếu với kiểu dữ liệu, % thiếu và giá trị thực.
- Cột *"biết lúc nộp đơn?"* chỉ là giả thuyết ban đầu. Bước 5 sẽ kiểm chứng bằng code, đặc biệt là rủi ro dùng nhầm cột chỉ xuất hiện sau khi khoản vay đã bắt đầu (rò rỉ thông tin - leakage).

In [ ]:
# Chỉ đọc dòng tiêu đề của file gốc để lấy danh sách cột (không tốn bộ nhớ)
ALL_COLUMNS = list(pd.read_csv(RAW_FILE, nrows=0).columns)
print("Số cột trong file gốc:", len(ALL_COLUMNS))

In [ ]:
# Ba nhãn dùng cho giả thuyết "biết được lúc nộp đơn?"
CO, KHONG, KR = "Có", "Không", "Không rõ"

**Từ điển - nhóm khoản vay**

In [ ]:
KHOAN_VAY = {
 "id": ("Mã khoản vay do Lending Club gán", KHONG),
 "member_id": ("Mã thành viên của người vay", KHONG),
 "url": ("Đường dẫn tới trang của khoản vay trên website Lending Club", KHONG),
 "loan_amnt": ("Số tiền người vay đề nghị vay (USD)", CO),
 "funded_amnt": ("Số tiền được duyệt/giải ngân cho khoản vay (USD)", KR),
 "funded_amnt_inv": ("Số tiền do nhà đầu tư cấp cho khoản vay (USD)", KR),
 "term": ("Kỳ hạn vay: 36 hoặc 60 tháng", CO),
 "int_rate": ("Lãi suất của khoản vay (%/năm)", CO),
 "installment": ("Số tiền trả góp mỗi tháng (USD)", CO),
 "grade": ("Hạng rủi ro A-G do Lending Club gán", CO),
 "sub_grade": ("Hạng rủi ro chi tiết A1-G5 do Lending Club gán", CO),
 "issue_d": ("Tháng khoản vay được giải ngân (dùng để sắp xếp theo thời gian)", KR),
 "loan_status": ("Trạng thái hiện tại của khoản vay (nguồn tạo nhãn)", KHONG),
 "pymnt_plan": ("Cờ cho biết khoản vay có kế hoạch trả nợ riêng hay không", KR),
 "purpose": ("Mục đích vay do người vay chọn", CO),
 "title": ("Tiêu đề khoản vay do người vay viết", CO),
 "desc": ("Mô tả khoản vay do người vay viết", CO),
 "initial_list_status": ("Trạng thái niêm yết ban đầu của khoản vay (w hoặc f)", KR),
 "application_type": ("Hồ sơ vay cá nhân (Individual) hay vay chung (Joint App)", CO),
 "policy_code": ("Mã chính sách: sản phẩm công khai hay không công khai", KR),
 "disbursement_method": ("Cách giải ngân: tiền mặt hay trả thẳng cho chủ nợ khác", KR),
}

**Từ điển - nhóm người vay**

In [ ]:
NGUOI_VAY = {
 "emp_title": ("Chức danh công việc do người vay nhập", CO),
 "emp_length": ("Số năm làm việc ở công việc hiện tại (dưới 1 năm đến 10+ năm)", CO),
 "home_ownership": ("Tình trạng nhà ở: thuê, thế chấp, sở hữu,...", CO),
 "annual_inc": ("Thu nhập hằng năm do người vay tự khai (USD)", CO),
 "verification_status": ("Thu nhập đã được Lending Club xác minh hay chưa", CO),
 "zip_code": ("3 chữ số đầu của mã bưu chính người vay", CO),
 "addr_state": ("Bang nơi người vay sinh sống", CO),
 "dti": ("Tỷ lệ nợ phải trả hằng tháng trên thu nhập hằng tháng (%), không tính thế chấp nhà và khoản vay này", CO),
}

**Từ điển - nhóm hồ sơ tín dụng (phần 1)**

In [ ]:
TIN_DUNG_1 = {
 "delinq_2yrs": ("Số lần trễ hạn từ 30 ngày trở lên trong 2 năm gần nhất", CO),
 "earliest_cr_line": ("Tháng mở tài khoản tín dụng đầu tiên", CO),
 "fico_range_low": ("Cận dưới của khoảng điểm FICO lúc nộp đơn", CO),
 "fico_range_high": ("Cận trên của khoảng điểm FICO lúc nộp đơn", CO),
 "inq_last_6mths": ("Số lần bị tra cứu tín dụng trong 6 tháng gần nhất", CO),
 "mths_since_last_delinq": ("Số tháng kể từ lần trễ hạn gần nhất", CO),
 "mths_since_last_record": ("Số tháng kể từ hồ sơ công khai tiêu cực gần nhất", CO),
 "open_acc": ("Số tài khoản tín dụng đang mở", CO),
 "pub_rec": ("Số hồ sơ công khai tiêu cực", CO),
 "revol_bal": ("Tổng dư nợ tín dụng quay vòng (USD)", CO),
 "revol_util": ("Phần trăm hạn mức tín dụng quay vòng đã sử dụng", CO),
 "total_acc": ("Tổng số tài khoản tín dụng từng có", CO),
 "collections_12_mths_ex_med": ("Số khoản bị đòi nợ trong 12 tháng, không tính nợ y tế", CO),
 "mths_since_last_major_derog": ("Số tháng kể từ lần vi phạm nghiêm trọng gần nhất (trễ từ 90 ngày)", CO),
 "acc_now_delinq": ("Số tài khoản đang trễ hạn", CO),
 "tot_coll_amt": ("Tổng số tiền từng bị đòi nợ (USD)", CO),
 "tot_cur_bal": ("Tổng dư nợ hiện tại của mọi tài khoản (USD)", CO),
 "open_acc_6m": ("Số tài khoản mở trong 6 tháng gần nhất", CO),
 "open_act_il": ("Số tài khoản trả góp đang hoạt động", CO),
 "open_il_12m": ("Số tài khoản trả góp mở trong 12 tháng gần nhất", CO),
 "open_il_24m": ("Số tài khoản trả góp mở trong 24 tháng gần nhất", CO),
 "mths_since_rcnt_il": ("Số tháng kể từ khi mở tài khoản trả góp gần nhất", CO),
 "total_bal_il": ("Tổng dư nợ các tài khoản trả góp (USD)", CO),
 "il_util": ("Tỷ lệ dư nợ trả góp trên hạn mức cao nhất", CO),
 "open_rv_12m": ("Số tài khoản quay vòng mở trong 12 tháng gần nhất", CO),
 "open_rv_24m": ("Số tài khoản quay vòng mở trong 24 tháng gần nhất", CO),
 "max_bal_bc": ("Dư nợ lớn nhất trên các tài khoản quay vòng (USD)", CO),
 "all_util": ("Tỷ lệ sử dụng trên tổng hạn mức của mọi tài khoản", CO),
 "total_rev_hi_lim": ("Tổng hạn mức cao nhất của các tài khoản quay vòng (USD)", CO),
 "inq_fi": ("Số lần tra cứu tín dụng của công ty tài chính cá nhân", CO),
 "total_cu_tl": ("Số tài khoản tài chính (finance trades)", CO),
 "inq_last_12m": ("Số lần bị tra cứu tín dụng trong 12 tháng gần nhất", CO),
 "acc_open_past_24mths": ("Số tài khoản mở trong 24 tháng gần nhất", CO),
 "avg_cur_bal": ("Dư nợ hiện tại trung bình mỗi tài khoản (USD)", CO),
 "bc_open_to_buy": ("Hạn mức còn lại trên các thẻ ngân hàng quay vòng (USD)", CO),
 "bc_util": ("Tỷ lệ sử dụng hạn mức trên các thẻ ngân hàng", CO),
 "chargeoff_within_12_mths": ("Số lần bị xóa nợ trong 12 tháng gần nhất", CO),
 "delinq_amnt": ("Số tiền đang trễ hạn (USD)", CO),
}

**Từ điển - nhóm hồ sơ tín dụng (phần 2)**

In [ ]:
TIN_DUNG_2 = {
 "mo_sin_old_il_acct": ("Số tháng kể từ khi mở tài khoản trả góp lâu đời nhất", CO),
 "mo_sin_old_rev_tl_op": ("Số tháng kể từ khi mở tài khoản quay vòng lâu đời nhất", CO),
 "mo_sin_rcnt_rev_tl_op": ("Số tháng kể từ khi mở tài khoản quay vòng gần nhất", CO),
 "mo_sin_rcnt_tl": ("Số tháng kể từ khi mở tài khoản bất kỳ gần nhất", CO),
 "mort_acc": ("Số tài khoản thế chấp nhà", CO),
 "mths_since_recent_bc": ("Số tháng kể từ khi mở thẻ ngân hàng gần nhất", CO),
 "mths_since_recent_bc_dlq": ("Số tháng kể từ lần trễ hạn thẻ ngân hàng gần nhất", CO),
 "mths_since_recent_inq": ("Số tháng kể từ lần bị tra cứu tín dụng gần nhất", CO),
 "mths_since_recent_revol_delinq": ("Số tháng kể từ lần trễ hạn tài khoản quay vòng gần nhất", CO),
 "num_accts_ever_120_pd": ("Số tài khoản từng trễ hạn từ 120 ngày", CO),
 "num_actv_bc_tl": ("Số thẻ ngân hàng đang hoạt động", CO),
 "num_actv_rev_tl": ("Số tài khoản quay vòng đang hoạt động", CO),
 "num_bc_sats": ("Số thẻ ngân hàng đang ở trạng thái tốt", CO),
 "num_bc_tl": ("Tổng số thẻ ngân hàng", CO),
 "num_il_tl": ("Số tài khoản trả góp", CO),
 "num_op_rev_tl": ("Số tài khoản quay vòng đang mở", CO),
 "num_rev_accts": ("Số tài khoản quay vòng", CO),
 "num_rev_tl_bal_gt_0": ("Số tài khoản quay vòng có dư nợ lớn hơn 0", CO),
 "num_sats": ("Số tài khoản đang ở trạng thái tốt", CO),
 "num_tl_120dpd_2m": ("Số tài khoản trễ hạn từ 120 ngày (cập nhật trong 2 tháng)", CO),
 "num_tl_30dpd": ("Số tài khoản đang trễ hạn 30 ngày (cập nhật trong 2 tháng)", CO),
 "num_tl_90g_dpd_24m": ("Số tài khoản trễ hạn từ 90 ngày trong 24 tháng gần nhất", CO),
 "num_tl_op_past_12m": ("Số tài khoản mở trong 12 tháng gần nhất", CO),
 "pct_tl_nvr_dlq": ("Phần trăm tài khoản chưa từng trễ hạn", CO),
 "percent_bc_gt_75": ("Phần trăm thẻ ngân hàng đã dùng trên 75% hạn mức", CO),
 "pub_rec_bankruptcies": ("Số lần phá sản trong hồ sơ công khai", CO),
 "tax_liens": ("Số lần bị cầm giữ tài sản vì thuế", CO),
 "tot_hi_cred_lim": ("Tổng hạn mức tín dụng cao nhất (USD)", CO),
 "total_bal_ex_mort": ("Tổng dư nợ, không tính thế chấp nhà (USD)", CO),
 "total_bc_limit": ("Tổng hạn mức các thẻ ngân hàng (USD)", CO),
 "total_il_high_credit_limit": ("Tổng hạn mức cao nhất của các tài khoản trả góp (USD)", CO),
}

**Từ điển - nhóm hồ sơ vay chung**

In [ ]:
VAY_CHUNG = {
 "annual_inc_joint": ("Thu nhập hằng năm gộp của hai người vay chung (USD)", CO),
 "dti_joint": ("Tỷ lệ nợ trên thu nhập gộp của hai người vay chung (%)", CO),
 "verification_status_joint": ("Thu nhập gộp của hai người vay chung đã được xác minh hay chưa", CO),
 "revol_bal_joint": ("Tổng dư nợ quay vòng gộp của hai người vay chung (USD)", CO),
 "sec_app_fico_range_low": ("Cận dưới điểm FICO của người vay thứ hai", CO),
 "sec_app_fico_range_high": ("Cận trên điểm FICO của người vay thứ hai", CO),
 "sec_app_earliest_cr_line": ("Tháng mở tài khoản tín dụng đầu tiên của người vay thứ hai", CO),
 "sec_app_inq_last_6mths": ("Số lần tra cứu tín dụng 6 tháng của người vay thứ hai", CO),
 "sec_app_mort_acc": ("Số tài khoản thế chấp của người vay thứ hai", CO),
 "sec_app_open_acc": ("Số tài khoản đang mở của người vay thứ hai", CO),
 "sec_app_revol_util": ("Phần trăm hạn mức quay vòng đã dùng của người vay thứ hai", CO),
 "sec_app_open_act_il": ("Số tài khoản trả góp đang hoạt động của người vay thứ hai", CO),
 "sec_app_num_rev_accts": ("Số tài khoản quay vòng của người vay thứ hai", CO),
 "sec_app_chargeoff_within_12_mths": ("Số lần bị xóa nợ 12 tháng của người vay thứ hai", CO),
 "sec_app_collections_12_mths_ex_med": ("Số khoản bị đòi nợ 12 tháng (trừ y tế) của người vay thứ hai", CO),
 "sec_app_mths_since_last_major_derog": ("Số tháng kể từ lần vi phạm nghiêm trọng gần nhất của người vay thứ hai", CO),
}

**Từ điển - nhóm thanh toán sau khi vay**

In [ ]:
SAU_KHI_VAY = {
 "out_prncp": ("Số tiền gốc còn nợ (USD)", KHONG),
 "out_prncp_inv": ("Số tiền gốc còn nợ phần do nhà đầu tư cấp (USD)", KHONG),
 "total_pymnt": ("Tổng số tiền đã thu được cho đến nay (USD)", KHONG),
 "total_pymnt_inv": ("Tổng số tiền đã thu được phần của nhà đầu tư (USD)", KHONG),
 "total_rec_prncp": ("Tổng tiền gốc đã thu được (USD)", KHONG),
 "total_rec_int": ("Tổng tiền lãi đã thu được (USD)", KHONG),
 "total_rec_late_fee": ("Tổng phí trả chậm đã thu được (USD)", KHONG),
 "recoveries": ("Số tiền thu hồi được sau khi khoản vay bị xóa nợ (USD)", KHONG),
 "collection_recovery_fee": ("Phí thu hồi nợ sau khi bị xóa nợ (USD)", KHONG),
 "last_pymnt_d": ("Tháng nhận được khoản thanh toán gần nhất", KHONG),
 "last_pymnt_amnt": ("Số tiền của khoản thanh toán gần nhất (USD)", KHONG),
 "next_pymnt_d": ("Tháng dự kiến của khoản thanh toán tiếp theo", KHONG),
 "last_credit_pull_d": ("Tháng Lending Club kéo hồ sơ tín dụng lần gần nhất", KHONG),
 "last_fico_range_high": ("Cận trên điểm FICO ở lần kéo hồ sơ tín dụng gần nhất", KHONG),
 "last_fico_range_low": ("Cận dưới điểm FICO ở lần kéo hồ sơ tín dụng gần nhất", KHONG),
}

**Từ điển - nhóm hỗ trợ khó khăn tài chính**

In [ ]:
KHO_KHAN = {
 "hardship_flag": ("Cờ cho biết người vay có kế hoạch hỗ trợ khó khăn tài chính hay không", KHONG),
 "hardship_type": ("Loại kế hoạch hỗ trợ khó khăn", KHONG),
 "hardship_reason": ("Lý do xin hỗ trợ khó khăn", KHONG),
 "hardship_status": ("Trạng thái kế hoạch hỗ trợ khó khăn", KHONG),
 "deferral_term": ("Số tháng được hoãn trả theo kế hoạch hỗ trợ", KHONG),
 "hardship_amount": ("Số tiền được hỗ trợ theo kế hoạch (USD)", KHONG),
 "hardship_start_date": ("Ngày bắt đầu kế hoạch hỗ trợ", KHONG),
 "hardship_end_date": ("Ngày kết thúc kế hoạch hỗ trợ", KHONG),
 "payment_plan_start_date": ("Ngày bắt đầu kế hoạch trả nợ mới", KHONG),
 "hardship_length": ("Số tháng của kế hoạch hỗ trợ", KHONG),
 "hardship_dpd": ("Số ngày trễ hạn tại thời điểm bắt đầu kế hoạch hỗ trợ", KHONG),
 "hardship_loan_status": ("Trạng thái khoản vay tại thời điểm bắt đầu kế hoạch hỗ trợ", KHONG),
 "orig_projected_additional_accrued_interest": ("Lãi dự kiến cộng thêm do kế hoạch hỗ trợ (USD)", KHONG),
 "hardship_payoff_balance_amount": ("Số dư cần trả hết khi bắt đầu kế hoạch hỗ trợ (USD)", KHONG),
 "hardship_last_payment_amount": ("Số tiền thanh toán cuối cùng trước khi bắt đầu kế hoạch hỗ trợ (USD)", KHONG),
}

**Từ điển - nhóm tất toán nợ**

In [ ]:
TAT_TOAN = {
 "debt_settlement_flag": ("Cờ cho biết khoản vay đã chuyển sang công ty tất toán nợ hay chưa", KHONG),
 "debt_settlement_flag_date": ("Ngày cờ tất toán nợ được cập nhật gần nhất", KHONG),
 "settlement_status": ("Trạng thái của thỏa thuận tất toán nợ", KHONG),
 "settlement_date": ("Ngày đạt thỏa thuận tất toán nợ", KHONG),
 "settlement_amount": ("Số tiền thỏa thuận tất toán (USD)", KHONG),
 "settlement_percentage": ("Phần trăm số dư mà người vay đồng ý trả khi tất toán", KHONG),
 "settlement_term": ("Số tháng của thỏa thuận tất toán", KHONG),
}

In [ ]:
GROUPS = [
    ("Thông tin khoản vay", KHOAN_VAY), ("Thông tin người vay", NGUOI_VAY),
    ("Hồ sơ tín dụng (phần 1)", TIN_DUNG_1), ("Hồ sơ tín dụng (phần 2)", TIN_DUNG_2),
    ("Hồ sơ vay chung", VAY_CHUNG), ("Thanh toán sau khi vay", SAU_KHI_VAY),
    ("Hỗ trợ khó khăn tài chính", KHO_KHAN), ("Tất toán nợ", TAT_TOAN),
]

Hàm dưới đây ghép tám nhóm thành một bảng từ điển duy nhất, mỗi cột của file gốc một dòng.

In [ ]:
def build_dictionary(groups):
    rows = []
    for group_name, entries in groups:
        for column, (meaning, known_at_application) in entries.items():
            rows.append({"column": column, "group": group_name, "meaning": meaning,
                         "known_at_application (giả thuyết)": known_at_application})
    return pd.DataFrame(rows).set_index("column")

In [ ]:
data_dictionary = build_dictionary(GROUPS)
print("Số dòng của từ điển:", len(data_dictionary))

Kiểm tra từ điển có **đúng** các cột của file gốc: không thiếu, không thừa, không trùng.

In [ ]:
missing_in_dictionary = [c for c in ALL_COLUMNS if c not in data_dictionary.index]
extra_in_dictionary = [c for c in data_dictionary.index if c not in ALL_COLUMNS]
print("Cột của file gốc chưa có trong từ điển:", missing_in_dictionary)
print("Cột trong từ điển mà file gốc không có:", extra_in_dictionary)
print("Số cột bị trùng trong từ điển:", data_dictionary.index.duplicated().sum())

In [ ]:
assert not missing_in_dictionary and not extra_in_dictionary
assert not data_dictionary.index.duplicated().any()
data_dictionary = data_dictionary.loc[ALL_COLUMNS]   # sắp xếp lại đúng thứ tự cột của file gốc
print("Từ điển khớp hoàn toàn với", len(ALL_COLUMNS), "cột của file gốc.")

**Bảng 1.** Số cột trong từng nhóm của từ điển.

In [ ]:
group_counts = data_dictionary.groupby("group", sort=False).size().to_frame("số cột")
group_counts.loc["Tổng"] = group_counts["số cột"].sum()
group_counts

**Bảng 2.** Số cột theo giả thuyết "biết được lúc nộp đơn" (mới là giả thuyết, chưa kiểm chứng).

In [ ]:
data_dictionary["known_at_application (giả thuyết)"].value_counts().to_frame("số cột")

**Bảng 3.** Toàn bộ từ điển dữ liệu.

In [ ]:
data_dictionary[["group", "meaning", "known_at_application (giả thuyết)"]]

---
# Bước 2 - Nạp dữ liệu gốc và quan sát ban đầu

## 2.1 Nạp toàn bộ file gốc, đủ mọi cột
Ta nạp nguyên file, **không chọn trước cột nào**, để mọi quyết định bỏ hay giữ cột sau này đều dựa trên bằng chứng. Ô sau đây mất vài phút vì file nén lớn.

In [ ]:
df_raw = pd.read_csv(RAW_FILE, low_memory=False)

In [ ]:
print("Số dòng, số cột:", df_raw.shape)

In [ ]:
memory_gb = df_raw.memory_usage(deep=True).sum() / 1e9
print(f"Bộ nhớ DataFrame chiếm: {memory_gb:.2f} GB")

In [ ]:
assert list(df_raw.columns) == ALL_COLUMNS
print("Các cột nạp vào đúng bằng các cột trong từ điển, cùng thứ tự.")

## 2.2 Đối chiếu từ điển với dữ liệu thật
Hàm dưới đây tính cho mỗi cột: kiểu dữ liệu, % giá trị thiếu, số giá trị khác nhau và một giá trị ví dụ.

In [ ]:
def describe_columns(df):
    example = df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_%": df.isna().mean() * 100,
        "n_unique": df.nunique(),
        "example": example,
    })

In [ ]:
column_overview = describe_columns(df_raw)

In [ ]:
print("Số cột theo kiểu dữ liệu:")
print(column_overview["dtype"].value_counts().to_string())

**Bảng 4.** Từ điển ghép với thống kê thật của từng cột.

In [ ]:
dictionary_check = data_dictionary[["group", "meaning"]].join(column_overview)
dictionary_check

In [ ]:
very_empty = column_overview[column_overview["missing_%"] > 90]
print(f"Số cột thiếu trên 90% giá trị: {len(very_empty)} / {len(column_overview)}")
print(f"Số cột không thiếu giá trị nào: {(column_overview['missing_%'] == 0).sum()}")
print("Các nhóm có nhiều cột thiếu trên 90% nhất:")
print(data_dictionary.loc[very_empty.index, "group"].value_counts().to_string())

Một số cột trong từ điển có miền giá trị rõ ràng (tỷ lệ phần trăm, điểm FICO, số tiền dương). Ta kiểm tra bằng số liệu thật xem mô tả có hợp lý không.

**Bảng 5.** Thống kê mô tả của vài cột số quan trọng.

In [ ]:
check_columns = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "revol_util",
                 "fico_range_low", "fico_range_high"]
df_raw[check_columns].describe().T

In [ ]:
print("Điểm FICO nằm trong khoảng 300-850:", df_raw["fico_range_low"].between(300, 850).all(),
      "(bỏ qua dòng thiếu:", df_raw["fico_range_low"].isna().sum(), "dòng)")
print("loan_amnt luôn dương:", (df_raw["loan_amnt"].dropna() > 0).all())
print("fico_range_low <= fico_range_high ở mọi dòng:", (df_raw["fico_range_low"] <= df_raw["fico_range_high"]).all())
print(f"revol_util lớn nhất: {df_raw['revol_util'].max():.1f}% (vượt 100% nghĩa là dùng quá hạn mức)")
print(f"dti lớn nhất: {df_raw['dti'].max():.1f}, annual_inc nhỏ nhất: {df_raw['annual_inc'].min():.1f}")
print("Hành động -> các giá trị bất thường (dti cực lớn, thu nhập bằng 0) sẽ được xử lý ở Bước 4, sau khi đã chia tập.")

## 2.3 Nhãn: xem `loan_status`

**Bảng 6.** Số khoản vay theo từng giá trị của `loan_status` (kể cả giá trị thiếu).

In [ ]:
status_table = df_raw["loan_status"].value_counts(dropna=False).to_frame("số khoản vay")
status_table["tỷ lệ %"] = status_table["số khoản vay"] / len(df_raw) * 100
status_table

In [ ]:
n_missing_status = df_raw["loan_status"].isna().sum()
print(f"Số dòng thiếu loan_status: {n_missing_status}")
print("Các dòng thiếu loan_status cũng thiếu cả ngày giải ngân issue_d:", df_raw.loc[df_raw["loan_status"].isna(), "issue_d"].isna().all())
print("Các dòng này không phải là khoản vay (không có thông tin gì) nên sẽ bị loại.")

Trạng thái nào là **kết quả cuối cùng** của khoản vay? Một khoản vay đã kết thúc thì không còn gốc nợ phải trả. Cột `out_prncp` là tiền gốc còn nợ, nên ta tính tỷ lệ khoản vay còn gốc nợ (> 0) theo từng trạng thái. Đây là cách dùng dữ liệu để tự kiểm chứng trạng thái nào đã kết thúc.

**Bảng 7.** Tỷ lệ khoản vay còn nợ gốc theo `loan_status`.

In [ ]:
outstanding = df_raw.groupby("loan_status")["out_prncp"].agg(
    so_khoan_vay="size", ty_le_con_no_goc=lambda s: (s > 0).mean())
outstanding

In [ ]:
finished = outstanding[outstanding["ty_le_con_no_goc"] == 0].index.tolist()
unfinished = outstanding[outstanding["ty_le_con_no_goc"] > 0].index.tolist()
print("Trạng thái không còn nợ gốc (đã kết thúc):", finished)
print("Trạng thái vẫn còn nợ gốc (chưa kết thúc):", unfinished)

Trong các trạng thái đã kết thúc có hai trạng thái ghi rõ là *Does not meet the credit policy*, tức là thuộc chính sách cho vay cũ. Để giữ bài toán đơn giản và đồng nhất sản phẩm vay, ta chỉ lấy hai trạng thái chuẩn `Fully Paid` (nhãn 0) và `Charged Off` (nhãn 1). Ô sau cho biết việc loại hai trạng thái chính sách cũ làm mất bao nhiêu dòng.

In [ ]:
old_policy = [s for s in finished if "credit policy" in s]
n_old_policy = df_raw["loan_status"].isin(old_policy).sum()
print(f"Số khoản vay đã kết thúc nhưng thuộc chính sách cũ (bị loại): {n_old_policy:,}")
print(f"Chiếm {n_old_policy / len(df_raw) * 100:.2f}% tổng số dòng.")

In [ ]:
LABEL_MAP = {"Fully Paid": 0, "Charged Off": 1}
df_raw["default"] = df_raw["loan_status"].map(LABEL_MAP)
print("Số dòng có nhãn:", df_raw["default"].notna().sum())

In [ ]:
label_counts = df_raw["default"].value_counts().rename({0: "0 = Fully Paid", 1: "1 = Charged Off"}).to_frame("số khoản vay")
label_counts["tỷ lệ %"] = label_counts["số khoản vay"] / label_counts["số khoản vay"].sum() * 100
label_counts

## 2.4 Đổi ngày giải ngân và kỳ hạn sang dạng số
Cột `issue_d` có dạng chữ như `Dec-2015`, cột `term` có dạng `" 36 months"`. Hai hàm nhỏ sau đổi chúng sang dạng dùng được.

In [ ]:
def parse_month(series):
    return pd.to_datetime(series, format="%b-%Y")

In [ ]:
def parse_term_months(series):
    return series.str.extract(r"(\d+)")[0].astype(float)

In [ ]:
df_raw["issue_date"] = parse_month(df_raw["issue_d"])
df_raw["term_months"] = parse_term_months(df_raw["term"])
df_raw["year"] = df_raw["issue_date"].dt.year

In [ ]:
print("Ngày giải ngân sớm nhất:", df_raw["issue_date"].min().date())
print("Ngày giải ngân muộn nhất:", df_raw["issue_date"].max().date())
print("Các kỳ hạn có trong dữ liệu:", sorted(df_raw["term_months"].dropna().unique()))

---
# Bước 3 - EDA phục vụ việc chia train / val / test

Mục tiêu của EDA ở đây **không phải** tìm xem feature nào dự đoán tốt (việc đó nếu làm trên toàn bộ dữ liệu sẽ vô tình "nhìn trước" vùng test). Ta chỉ hỏi các câu về **cấu trúc thời gian** của dữ liệu để quyết định chia thế nào.

**Tham khảo cách làm của người khác** (nêu tên và kỹ thuật; chưa đối chiếu nội dung gốc, mọi điều rút ra cho dữ liệu của ta đều do code bên dưới kiểm chứng):
- Arturo-GA, `lendingclub-default-risk` - https://github.com/Arturo-GA/lendingclub-default-risk : chia theo thời gian 70/15/15, chỉ dùng khoản vay đã đáo hạn.
- vaibhavkev, `credit-risk` - https://github.com/vaibhavkev/credit-risk : chia train/validation/test theo thời gian.
- Xia và cộng sự, SurvXGBoost - https://journals.vilniustech.lt/index.php/TEDE/article/view/13997 : đánh giá cả out-of-sample lẫn out-of-time.

## 3.1 Số khoản vay, tỷ lệ vỡ nợ và tỷ lệ đã có kết quả theo năm

In [ ]:
df_raw["resolved"] = df_raw["default"].notna()

In [ ]:
by_year = df_raw.groupby("year").agg(
    so_khoan_vay=("resolved", "size"),
    ty_le_da_co_ket_qua=("resolved", "mean"),
    ty_le_vo_no_trong_nhom_da_co_ket_qua=("default", "mean"),
    ty_le_ky_han_60_thang=("term_months", lambda s: (s == 60).mean()),
)
by_year.index = by_year.index.astype(int).astype(str)
by_year

In [ ]:
low_year = by_year["ty_le_da_co_ket_qua"].idxmin()
print(f"Năm có tỷ lệ khoản vay đã có kết quả thấp nhất: {low_year} ({by_year.loc[low_year, 'ty_le_da_co_ket_qua']:.1%}).")
print(f"Năm có tỷ lệ vỡ nợ cao nhất: {by_year['ty_le_vo_no_trong_nhom_da_co_ket_qua'].idxmax()} "
      f"({by_year['ty_le_vo_no_trong_nhom_da_co_ket_qua'].max():.3f}); thấp nhất: "
      f"{by_year['ty_le_vo_no_trong_nhom_da_co_ket_qua'].idxmin()} ({by_year['ty_le_vo_no_trong_nhom_da_co_ket_qua'].min():.3f}).")
print("Hành động -> các khoản vay gần cuối dữ liệu chưa kịp kết thúc nên nhãn bị lệch; cần một luật loại khoản vay chưa đáo hạn (mục 3.3).")

## 3.2 Bắt đầu từ năm nào?
Các năm đầu có rất ít khoản vay. Ta xem mỗi năm chiếm bao nhiêu phần trăm số khoản vay có nhãn, và các cột hồ sơ tín dụng bắt đầu có dữ liệu từ năm nào.

In [ ]:
labelled_per_year = df_raw[df_raw["resolved"]].groupby("year").size()
share_per_year = (labelled_per_year / labelled_per_year.sum() * 100).to_frame("tỷ lệ % trong số khoản vay có nhãn")
share_per_year.index = share_per_year.index.astype(int).astype(str)
share_per_year["số khoản vay có nhãn"] = labelled_per_year.values
share_per_year

Hàm sau trả về năm đầu tiên một cột có dữ liệu (không thiếu).

In [ ]:
def first_year_with_data(df, column):
    return df.loc[df[column].notna(), "year"].min()

In [ ]:
float_columns = [c for c in ALL_COLUMNS if df_raw[c].dtype == "float64"]
first_years = pd.Series({c: first_year_with_data(df_raw, c) for c in float_columns})

In [ ]:
first_year_counts = first_years.astype(int).value_counts().sort_index().to_frame("số cột số bắt đầu có dữ liệu từ năm này")
first_year_counts.index = first_year_counts.index.astype(str)
first_year_counts

In [ ]:
START_YEAR = 2010    # theo cách làm của Arturo-GA
before = labelled_per_year[labelled_per_year.index < START_YEAR].sum()
print(f"Các năm trước {START_YEAR} chỉ có {before:,} khoản vay có nhãn, chiếm {before / labelled_per_year.sum():.2%} tổng số.")
late_columns = (first_years > START_YEAR).sum()
print(f"Có {late_columns} cột số chỉ bắt đầu có dữ liệu sau năm {START_YEAR}.")
print(f"Quyết định -> bắt đầu từ năm {START_YEAR}: bỏ đi rất ít dữ liệu. Việc nhiều cột bắt đầu có dữ liệu muộn sẽ được xét ở Bước 5.")

## 3.3 Luật "đã đáo hạn"
Khoản vay phát hành năm 2018 với kỳ hạn 36 tháng chưa thể kết thúc. Nếu giữ lại, nhãn của chúng thiên về những khoản kết thúc sớm. Ta định nghĩa: một khoản vay **đã đáo hạn** nếu tháng giải ngân cộng kỳ hạn không muộn hơn tháng cuối cùng của dữ liệu.

Tháng cuối cùng của dữ liệu lấy từ chính dữ liệu, không gõ tay:

In [ ]:
SNAPSHOT = df_raw["issue_date"].max()
print("Tháng cuối cùng của dữ liệu (SNAPSHOT):", SNAPSHOT.strftime("%Y-%m"))

In [ ]:
def month_number(dates):
    return dates.dt.year * 12 + dates.dt.month

In [ ]:
def is_matured(issue_date, term_months, snapshot):
    end_month = month_number(issue_date) + term_months
    return end_month <= snapshot.year * 12 + snapshot.month

In [ ]:
df_raw["matured"] = is_matured(df_raw["issue_date"], df_raw["term_months"], SNAPSHOT)
print("Số khoản vay đã đáo hạn:", df_raw["matured"].sum())

**Kiểm chứng luật.** Nếu luật đúng thì trong nhóm đã đáo hạn, hầu hết khoản vay phải đã có kết quả cuối cùng; còn trong nhóm chưa đáo hạn, tỷ lệ đó phải thấp hơn nhiều.

**Bảng 8.** Tỷ lệ đã có kết quả cuối cùng, theo nhóm đáo hạn.

In [ ]:
maturity_check = df_raw.groupby("matured").agg(
    so_khoan_vay=("resolved", "size"), ty_le_da_co_ket_qua=("resolved", "mean"))
maturity_check.index = maturity_check.index.map({True: "đã đáo hạn", False: "chưa đáo hạn"})
maturity_check

In [ ]:
rate_matured = maturity_check.loc["đã đáo hạn", "ty_le_da_co_ket_qua"]
rate_not = maturity_check.loc["chưa đáo hạn", "ty_le_da_co_ket_qua"]
print(f"Nhóm đã đáo hạn: {rate_matured:.1%} có kết quả cuối cùng. Nhóm chưa đáo hạn: {rate_not:.1%}.")
print("Luật phân biệt được hai nhóm." if rate_matured > rate_not else "Luật KHÔNG phân biệt được hai nhóm - phải xem lại.")
print(f"Hành động -> giữ nhóm đã đáo hạn. {1 - rate_matured:.1%} khoản vay đã đáo hạn vẫn chưa có kết quả (trễ hạn, tạm hoãn...) sẽ bị loại ở bước lọc có nhãn.")

## 3.4 Các bộ lọc và số dòng còn lại
Ba bộ lọc: có nhãn, từ năm bắt đầu, đã đáo hạn. Bảng cho thấy mỗi bộ lọc loại bao nhiêu dòng.

**Bảng 9.** Số dòng sau từng bộ lọc.

In [ ]:
is_labelled = df_raw["default"].notna()
from_start_year = df_raw["year"] >= START_YEAR
is_old_enough = df_raw["matured"]

In [ ]:
steps = [
    ("Toàn bộ file gốc", pd.Series(True, index=df_raw.index)),
    ("Có nhãn (Fully Paid / Charged Off)", is_labelled),
    (f"... và phát hành từ năm {START_YEAR}", is_labelled & from_start_year),
    ("... và đã đáo hạn", is_labelled & from_start_year & is_old_enough),
]
filter_table = pd.DataFrame({"số dòng": [mask.sum() for _, mask in steps]}, index=[name for name, _ in steps])
filter_table["bị loại ở bước này"] = -filter_table["số dòng"].diff().fillna(0)
filter_table

In [ ]:
final_mask = is_labelled & from_start_year & is_old_enough
loans = df_raw[final_mask].copy()
print(f"Tập khoản vay dùng cho project: {len(loans):,} dòng ({len(loans) / len(df_raw):.1%} file gốc).")
print(f"Tỷ lệ vỡ nợ trong tập này: {loans['default'].mean():.3f}")
print(f"Ngày giải ngân từ {loans['issue_date'].min():%Y-%m} đến {loans['issue_date'].max():%Y-%m}.")

## 3.5 Chia 70 / 15 / 15 theo thời gian
Hàm `time_split` sắp các khoản vay theo ngày giải ngân, tìm hai **ngày cắt** tại vị trí 70% và 85% rồi chia theo ngày. Chia theo ngày (không phải theo vị trí dòng) để không có tháng nào nằm ở hai tập cùng lúc.

In [ ]:
def time_split(df, train_share=0.70, val_share=0.15):
    df = df.sort_values(["issue_date", "id"]).reset_index(drop=True)
    cut_val = df["issue_date"].iloc[int(len(df) * train_share)]
    cut_test = df["issue_date"].iloc[int(len(df) * (train_share + val_share))]
    train = df[df["issue_date"] < cut_val]
    val = df[(df["issue_date"] >= cut_val) & (df["issue_date"] < cut_test)]
    test = df[df["issue_date"] >= cut_test]
    return train, val, test, cut_val, cut_test

In [ ]:
train, val, test, CUT_VAL, CUT_TEST = time_split(loans, 0.70, 0.15)
print("Ngày bắt đầu tập val :", CUT_VAL.date())
print("Ngày bắt đầu tập test:", CUT_TEST.date())

In [ ]:
assert train["issue_date"].max() < val["issue_date"].min()
assert val["issue_date"].max() < test["issue_date"].min()
assert len(train) + len(val) + len(test) == len(loans)
print("Kiểm tra qua: train < val < test về thời gian, không mất và không trùng dòng nào.")

**Bảng 10.** Ba tập sau khi chia.

In [ ]:
split_table = pd.DataFrame({
    "số dòng": [len(train), len(val), len(test)],
    "tỷ lệ % tổng": [len(x) / len(loans) * 100 for x in (train, val, test)],
    "từ tháng": [x["issue_date"].min().strftime("%Y-%m") for x in (train, val, test)],
    "đến tháng": [x["issue_date"].max().strftime("%Y-%m") for x in (train, val, test)],
    "tỷ lệ vỡ nợ": [x["default"].mean() for x in (train, val, test)],
}, index=["train", "val", "test"])
split_table

**Biểu đồ 1.** Số khoản vay (cột) và tỷ lệ vỡ nợ (đường) theo tháng giải ngân, kèm hai đường cắt giữa train / val / test.

In [ ]:
monthly = loans.groupby("issue_date").agg(so_khoan_vay=("default", "size"), ty_le_vo_no=("default", "mean"))

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(monthly.index, monthly["so_khoan_vay"], width=25, color="lightsteelblue")
ax1.set_ylabel("Số khoản vay")
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["ty_le_vo_no"], color="firebrick")
ax2.set_ylabel("Tỷ lệ vỡ nợ")
for cut in (CUT_VAL, CUT_TEST):
    ax1.axvline(cut, color="black", linestyle="--")
ax1.set_title("Số khoản vay và tỷ lệ vỡ nợ theo tháng giải ngân")
plt.savefig(FIG_DIR / "01_chia_theo_thoi_gian.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rates = split_table["tỷ lệ vỡ nợ"]
print(f"Tỷ lệ vỡ nợ: train {rates['train']:.3f}, val {rates['val']:.3f}, test {rates['test']:.3f}.")
print(f"Tháng có nhiều khoản vay nhất: {monthly['so_khoan_vay'].idxmax():%Y-%m} ({monthly['so_khoan_vay'].max():,} khoản).")
trend = "tăng" if rates["test"] > rates["train"] else "giảm"
print(f"Tỷ lệ vỡ nợ {trend} từ train sang test ({rates['train']:.3f} -> {rates['test']:.3f}): đây là độ lệch (drift) theo thời gian "
      "mà mô hình sẽ phải đối mặt, đúng với cách dùng thực tế (dự đoán khoản vay tương lai từ khoản vay quá khứ).")

## 3.6 Lưu ba tập và bản ghi cách chia
Ba tập được lưu giữ nguyên đủ 151 cột gốc (cộng thêm vài cột phụ vừa tạo) để Notebook 2 dùng. Bản ghi `split_manifest.json` có số dòng, ngày cắt và dấu vân tay MD5 của các `id` trong tập train, để sau này kiểm tra được ai đó khác có cùng tập train hay không.

In [ ]:
train.to_parquet(OUT_DIR / "train.parquet", index=False)
val.to_parquet(OUT_DIR / "val.parquet", index=False)
test.to_parquet(OUT_DIR / "test.parquet", index=False)
print("Đã lưu train, val, test vào", OUT_DIR.resolve())

In [ ]:
manifest = {
    "quy_tac": "khoản vay có nhãn, phát hành từ START_YEAR, đã đáo hạn; sắp theo issue_d; chia 70/15/15 theo ngày",
    "start_year": START_YEAR,
    "snapshot": SNAPSHOT.strftime("%Y-%m"),
    "cut_val": str(CUT_VAL.date()),
    "cut_test": str(CUT_TEST.date()),
    "so_dong": {"train": len(train), "val": len(val), "test": len(test)},
    "ty_le_vo_no": {"train": round(float(rates["train"]), 4), "val": round(float(rates["val"]), 4), "test": round(float(rates["test"]), 4)},
    "md5_id_train": hashlib.md5(",".join(sorted(train["id"])).encode()).hexdigest(),
}
(ART_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(json.dumps(manifest, indent=2, ensure_ascii=False))